# EXP-007 (nhánh A) — Prompt lớp mô tả bằng AU
Không train. Thử 3 kiểu prompt mới (`AU_prompt_list` trong `models/Text.py`, mapping EMFACS cố định trước khi xem số):
- **AU Description Five**: chỉ mô tả AU, không có từ cảm xúc (VD "a photo of a face with raised cheeks and lip corners pulled up.")
- **Name with AU Five** *(prompt chính, chọn trước)*: tên cảm xúc + AU ("... an expression of happiness, with raised cheeks ...")
- **Ensemble Five with AU**: Ensemble Five gốc + Name with AU (10 câu/lớp)

So với **Expression Ensemble Five** trên cùng checkpoint: CLIP gốc · Paper · EXP-000 (3 seed) · EXP-004 λ=1 ep30 (3 seed).
Feature CLIP của ảnh/text tính 1 lần; mỗi head chỉ là phép nhân ma trận. Tiền xử lý giống `test.py`.

Tiêu chí: UAR trung bình 4 dataset tăng ≥ +1.0 so với Ensemble Five, không dataset nào giảm > 1.0.

# Cấu hình

In [3]:
SEEDS = [1, 2, 3]
BRANCH = "exp/007-au-instruction-prompts"   # chỉ cần models/clip + models/Text.py (không cần lavis)
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp007"
EXP000_ROOT = "/content/drive/MyDrive/ExpCLIP_output"  # tìm đệ quy expclip_l14_face_seed{N}-model.pth
EXP004_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
DATASETS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
N_CLS = {"RAFDB": 7, "AffectNet7": 7, "AffectNet8": 8, "FERPlus": 8}
BASE_PROMPT = "Expression Ensemble Five"
MAIN_PROMPT = "Name with AU Five"

# Mount Drive & GPU

In [4]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


# Clone repo & giải nén test set

In [5]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
!cd /content/Exp-CLIP && git checkout $BRANCH && git pull && git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/static_test_sets
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/static_test_sets/*/test
# CLIP vendored cần ftfy và pkg_resources (setuptools mới đã bỏ)
!pip install -q ftfy "setuptools<81"

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 437, done.
remote: Counting objects: 100% (437/437), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 437 (delta 319), reused 387 (delta 270), pack-reused 0 (from 0)
Receiving objects: 100% (437/437), 32.29 MiB | 20.37 MiB/s, done.
Resolving deltas: 100% (319/319), done.
Already on 'exp/007-au-instruction-prompts'
Your branch is up to date with 'origin/exp/007-au-instruction-prompts'.
Already up to date.
9a885f2 (HEAD -> exp/007-au-instruction-prompts, origin/exp/007-au-instruction-prompts) EXP-007: AU-described class prompts + test notebook
/content/static_test_sets/AffectNet7/test:
0_Happiness  1_Sadness	2_Neutral  3_Anger  4_Surprise	5_Disgust  6_Fear

/content/static_test_sets/AffectNet8/test:
0_Happiness  2_Neutral	4_Surprise  6_Fear
1_Sadness    3_Anger	5_Disgust   7_Contempt

/content/static_test_sets/FERPlus/test:
0_Happiness  2_Neutral	4_Surprise  6_Fear
1_Sadness    3_Anger	5_Disgust   7_Co

# Tìm checkpoint
Thiếu file nào thì bỏ qua (in cảnh báo).

In [6]:
import glob

def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

HEADS = {
    "CLIP gốc": {0: None},
    "Paper": {0: "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"},
    "EXP-000": {s: find_ckpt(EXP000_ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
    "EXP-004": {s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e30-model.pth" for s in SEEDS},
}
for m, d in HEADS.items():
    for s, p in list(d.items()):
        if p is not None and not os.path.exists(p):
            print(f"CẢNH BÁO: {m} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]
        else:
            print(f"{m} seed {s}: {p}")

CLIP gốc seed 0: None
Paper seed 0: /content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth
EXP-000 seed 1: /content/drive/MyDrive/ExpCLIP_output/expclip_l14_face_seed1-model.pth
EXP-000 seed 2: /content/drive/MyDrive/ExpCLIP_output/expclip_l14_face_seed2-model.pth
EXP-000 seed 3: /content/drive/MyDrive/ExpCLIP_output/expclip_l14_face_seed3-model.pth
EXP-004 seed 1: /content/drive/MyDrive/ExpCLIP_output/exp004/exp004_anchor_crossA_l1_l14_seed1-e30-model.pth
EXP-004 seed 2: /content/drive/MyDrive/ExpCLIP_output/exp004/exp004_anchor_crossA_l1_l14_seed2-e30-model.pth
EXP-004 seed 3: /content/drive/MyDrive/ExpCLIP_output/exp004/exp004_anchor_crossA_l1_l14_seed3-e30-model.pth


# Feature CLIP L/14 (ảnh 4 dataset + text 8 kiểu prompt), tính 1 lần

In [7]:
import sys, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets, torchvision.transforms as T
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list, FER_prompt_type_list, AU_prompt_list, AU_prompt_type_list

PROMPTS = dict(zip(FER_prompt_type_list + AU_prompt_type_list, FER_prompt_list + AU_prompt_list))
PROMPT_NAMES = list(PROMPTS)
clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()

img, y, classes, txt = {}, {}, {}, {}
with torch.no_grad():
    for ds_name in DATASETS:
        ds = datasets.ImageFolder(f"/content/static_test_sets/{ds_name}/test",
                                  T.Compose([T.Resize((224, 224)), T.ToTensor()]))   # giống test.py
        loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
        img[ds_name] = torch.cat([clip_model.encode_image(x.cuda()).float().cpu() for x, _ in loader])
        y[ds_name] = np.array(ds.targets)
        classes[ds_name] = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
        for pn, pl in PROMPTS.items():
            txt[(ds_name, pn)] = clip_model.encode_text(
                clip.tokenize(pl[ds_name], context_length=77, truncate=True).cuda()).float().cpu()
        print(ds_name, tuple(img[ds_name].shape), classes[ds_name])

/content/Exp-CLIP/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|████████████████████████████████████████| 890M/890M [00:05<00:00, 178MiB/s]


RAFDB (3068, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet7 (3500, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet8 (4000, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']
FERPlus (3573, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']


# Dự đoán & metric

In [8]:
def load_W(path):
    if path is None:
        return None
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def predict(W, ds_name, pn):
    i, t = img[ds_name], txt[(ds_name, pn)]
    if W is not None:
        i, t = i @ W.T, t @ W.T
    i, t = F.normalize(i, dim=-1), F.normalize(t, dim=-1)
    return (i @ t.T).view(len(i), N_CLS[ds_name], -1).mean(-1).argmax(1).numpy()   # prompt xếp theo lớp, như test.py

def metrics(y_, p_, n_cls):
    acc = np.array([(p_[y_ == c] == c).mean() * 100 for c in range(n_cls)])
    return {"UAR": acc.mean(), "WAR": (p_ == y_).mean() * 100, "acc": acc}

rows = []
for m, d in HEADS.items():
    for s, path in d.items():
        W = load_W(path)
        for ds_name in DATASETS:
            for pn in PROMPT_NAMES:
                r = metrics(y[ds_name], predict(W, ds_name, pn), N_CLS[ds_name])
                rows.append({"model": m, "seed": s, "dataset": ds_name, "prompt": pn, "UAR": r["UAR"], "WAR": r["WAR"],
                             **{f"acc_{c}": a for c, a in zip(classes[ds_name], r["acc"])}})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp007_au_prompts-results.csv", index=False)
print(len(res), "dòng →", OUT_DIR)

256 dòng → /content/drive/MyDrive/ExpCLIP_output/exp007


# Kiểm tra: Ensemble Five phải khớp `test.py`
So với kết quả đã ghi của EXP-000 / EXP-004. Lệch ≤ ~0.1 là do GPU fp16; lệch lớn thì pipeline sai, dừng lại.

In [9]:
REF = {"EXP-000|1|RAFDB": [58.88, 66.88], "EXP-000|1|AffectNet7": [44.09, 44.09], "EXP-000|1|AffectNet8": [39.23, 39.23], "EXP-000|1|FERPlus": [40.65, 51.47], "EXP-000|2|RAFDB": [57.82, 65.78], "EXP-000|2|AffectNet7": [43.69, 43.69], "EXP-000|2|AffectNet8": [38.17, 38.17], "EXP-000|2|FERPlus": [41.08, 53.6], "EXP-000|3|RAFDB": [57.5, 63.3], "EXP-000|3|AffectNet7": [44.74, 44.74], "EXP-000|3|AffectNet8": [39.12, 39.12], "EXP-000|3|FERPlus": [42.93, 55.16], "EXP-004|1|RAFDB": [58.73, 65.94], "EXP-004|1|AffectNet7": [45.83, 45.83], "EXP-004|1|AffectNet8": [39.6, 39.6], "EXP-004|1|FERPlus": [46.14, 52.03], "EXP-004|2|RAFDB": [57.07, 64.77], "EXP-004|2|AffectNet7": [43.97, 43.97], "EXP-004|2|AffectNet8": [38.75, 38.75], "EXP-004|2|FERPlus": [45.63, 52.45], "EXP-004|3|RAFDB": [58.26, 65.06], "EXP-004|3|AffectNet7": [45.46, 45.46], "EXP-004|3|AffectNet8": [39.07, 39.08], "EXP-004|3|FERPlus": [45.34, 51.44]}

chk = []
for k, (u, w) in REF.items():
    m, s, ds_name = k.split("|")
    r = res[(res.model == m) & (res.seed == int(s)) & (res.dataset == ds_name) & (res.prompt == BASE_PROMPT)]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u, "dWAR": r.WAR.item() - w})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(5).to_string(index=False))

24/24 cấu hình so được | max |ΔUAR| = 0.39, max |ΔWAR| = 0.08
                 key  dUAR  dWAR
   EXP-000|2|FERPlus  0.39 -0.00
     EXP-000|2|RAFDB  0.11  0.03
     EXP-004|2|RAFDB  0.09  0.06
     EXP-000|1|RAFDB -0.08 -0.06
EXP-004|2|AffectNet7  0.06  0.06


# Bảng 1 — UAR theo prompt × dataset (mean ± std qua seed)

In [10]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

res["Mean4"] = res.groupby(["model", "seed", "prompt"]).UAR.transform("mean")
for m in HEADS:
    r = res[res.model == m]
    t = r.pivot_table(index="prompt", columns="dataset", values="UAR", aggfunc=fmt)[DATASETS]
    t["Mean4"] = r.drop_duplicates(["seed", "prompt"]).groupby("prompt").Mean4.apply(fmt)
    print(f"--- {m}\n{t.loc[PROMPT_NAMES].to_string()}\n")

--- CLIP gốc
dataset                                       RAFDB AffectNet7 AffectNet8 FERPlus  Mean4
prompt                                                                                  
Class Name                                    48.16      39.03      33.12   38.17  39.62
An Expression of Name                         49.54      39.94      36.75   32.70  39.73
A Photo of A Face with An Expression of Name  47.38      36.00      31.25   29.11  35.93
Expression Ensemble Five                      51.90      40.77      36.45   39.00  42.03
Expression Ensemble Ten                       53.99      41.69      35.27   38.08  42.26
AU Description Five                           23.14      26.34      23.10   21.33  23.48
Name with AU Five                             50.78      44.69      39.60   44.17  44.81
Ensemble Five with AU                         52.87      44.46      39.28   43.26  44.97

--- Paper
dataset                                       RAFDB AffectNet7 AffectNet8 FERPlus  Mea

# Bảng 2 — Δ UAR so với Ensemble Five (cùng checkpoint, mean qua seed)
Tiêu chí: Mean4 ≥ +1.0 và không dataset nào < −1.0.

In [11]:
u = res.pivot_table(index=["model", "seed", "prompt"], columns="dataset", values="UAR")[DATASETS]
u["Mean4"] = u.mean(1)
for m in HEADS:
    base = u.loc[m].xs(BASE_PROMPT, level="prompt")
    d = pd.concat({pn: u.loc[m].xs(pn, level="prompt") - base for pn in PROMPT_NAMES if pn != BASE_PROMPT})
    t = d.groupby(level=0).agg(fmt).loc[[pn for pn in PROMPT_NAMES if pn != BASE_PROMPT]]
    print(f"--- {m}\n{t.to_string()}\n")

--- CLIP gốc
dataset                                        RAFDB AffectNet7 AffectNet8 FERPlus   Mean4
Class Name                                     -3.74      -1.74      -3.33   -0.82   -2.41
An Expression of Name                          -2.36      -0.83       0.30   -6.30   -2.30
A Photo of A Face with An Expression of Name   -4.52      -4.77      -5.20   -9.89   -6.10
Expression Ensemble Ten                         2.09       0.91      -1.18   -0.91    0.23
AU Description Five                           -28.76     -14.43     -13.35  -17.67  -18.55
Name with AU Five                              -1.12       3.91       3.15    5.18    2.78
Ensemble Five with AU                           0.97       3.69       2.83    4.26    2.93

--- Paper
dataset                                        RAFDB AffectNet7 AffectNet8 FERPlus   Mean4
Class Name                                     -7.45      -2.00      -2.05   -0.88   -3.09
An Expression of Name                          -0.12      -0.26   

# Bảng 3 — Accuracy từng lớp: Name with AU vs Ensemble Five (mean seed)
Lớp nào được lợi / bị hại khi thêm AU?

In [12]:
for ds_name in ["RAFDB", "FERPlus"]:
    cols = [f"acc_{c}" for c in classes[ds_name]]
    for m in ["EXP-000", "EXP-004"]:
        r = res[(res.model == m) & (res.dataset == ds_name) & res.prompt.isin([BASE_PROMPT, MAIN_PROMPT, "AU Description Five"])]
        t = r.groupby("prompt")[cols].mean()
        t.columns = classes[ds_name]
        t.loc["Δ Name+AU"] = t.loc[MAIN_PROMPT] - t.loc[BASE_PROMPT]
        print(f"--- {m} / {ds_name}\n{t.round(1).to_string()}\n")

--- EXP-000 / RAFDB
                          Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
prompt                                                                               
AU Description Five            62.4      1.5      9.3   16.7      32.1     55.8   4.5
Expression Ensemble Five       87.3     45.7     45.2   88.9      60.2     50.0  29.3
Name with AU Five              93.6     24.0     13.6   53.1      58.1     77.3  31.1
Δ Name+AU                       6.3    -21.7    -31.6  -35.8      -2.1     27.3   1.8

--- EXP-004 / RAFDB
                          Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
prompt                                                                               
AU Description Five            83.4      4.7      4.4   16.5      20.2     50.6  15.8
Expression Ensemble Five       89.3     52.6     33.0   91.4      70.3     42.7  27.0
Name with AU Five              96.8     57.2     20.2   62.3      60.5     57.5  29.7
Δ Name+AU    

# Lưu JSON (cùng định dạng `research_log/results/`)

In [13]:
out = {"source": "notebooks/EXP_007_au_prompts_test.ipynb",
       "rows": [{"model": r.model, "seed": int(r.seed), "prompt": r.prompt, "dataset": r.dataset,
                 "UAR": round(r.UAR, 2), "WAR": round(r.WAR, 2)} for r in res.itertuples()]}
with open(f"{OUT_DIR}/EXP-007-au-prompts.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"{len(out['rows'])} dòng → {OUT_DIR}/EXP-007-au-prompts.json")

256 dòng → /content/drive/MyDrive/ExpCLIP_output/exp007/EXP-007-au-prompts.json
